In [1]:
# This notebook use the jsonl data shared by the instructor, and create a new database

In [2]:

import json
import time

import psycopg
from psycopg import OperationalError

In [ ]:
# def load_data(filename):
#     """
#     Loads cleaned data from a JSON file.
#     """
#     with open(filename, "r", encoding="utf-8") as f:
#         data = json.load(f)
#     return data

In [ ]:
# directory = '/Users/jennifer/Documents/software_concept_python_class/jhu_software_concepts/Module_2/'

# applicant_data= load_data(directory+ "applicant_data_v2.json")
# len(applicant_data)

41000

In [5]:
def load_data_jsonl(filename):
    """
    Loads data from a JSONL file. Each line is one JSON object.
    """
    data = []

    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

# back up version of the data
applicant_data= load_data_jsonl("llm_extend_applicant_data_run.jsonl")
len(applicant_data)

36000

In [6]:
type(applicant_data)

list

In [7]:
# inspect the data

print(applicant_data[0])


{'program': 'Environmental Economics, Stockholm University', 'university_raw': 'Stockholm University', 'program_raw': 'Environmental Economics', 'comments': '', 'date_added': 'May 31, 2026', 'url': 'https://www.thegradcafe.com/result/1020288', 'status': 'Accepted on May 31', 'acceptance_date': 'May 31', 'rejection_date': None, 'term': 'Fall 2026', 'US/International': 'International', 'Degree': 'PhD', 'GPA': None, 'GRE': None, 'GRE V': None, 'GRE AW': None, 'llm-generated-program': 'Environmental Economics', 'llm-generated-university': 'Stockholm University'}


In [8]:
print(applicant_data[0].keys()) 

dict_keys(['program', 'university_raw', 'program_raw', 'comments', 'date_added', 'url', 'status', 'acceptance_date', 'rejection_date', 'term', 'US/International', 'Degree', 'GPA', 'GRE', 'GRE V', 'GRE AW', 'llm-generated-program', 'llm-generated-university'])


In [9]:
# Create connection to system database
def create_system_connection(user, password):
    """Connect to postgres system database"""
    try:
        conn = psycopg.connect(
            dbname="postgres",
            user=user,
            password=password,
            host="localhost"
        )
        conn.autocommit = True
        return conn
    except OperationalError as e:
        print(f"Connection failed: {e}")
        return None

In [11]:
# Create new database
def create_database(conn, db_name):
    """Create a new database"""
    try:
        cur = conn.cursor()
        cur.execute(f"CREATE DATABASE {db_name};")
        cur.close()
        print(f"Database '{db_name}' created successfully")
    except OperationalError as e:
        print(f"Database creation failed: {e}")

In [12]:
# created the database gradcafe_db_v2

system_conn = create_system_connection("postgres", "181818")
create_database(system_conn, "gradcafe_db_v2")
system_conn.close()

Database 'gradcafe_db_v2' created successfully


In [13]:
def create_db_connection(db_name, db_user, db_password, db_host, db_port):

    """
    create conenction to my db
    """

    connection = None
    try:
        connection = psycopg.connect(
            dbname=db_name,
            user=db_user,
            password=db_password,
            host=db_host,
            port=db_port,
        )
        print("Connection to PostgreSQL DB successful")
    except OperationalError as e:
        print(f"The error '{e}' occurred")
    return connection

In [14]:
connection = create_db_connection("gradcafe_db_v2", "postgres", "181818", "127.0.0.1", "5432")

Connection to PostgreSQL DB successful


In [15]:
def execute_query(connection, query):
    connection.autocommit = True
    cursor = connection.cursor()
    try:
        cursor.execute(query)
        # print("Query executed successfully")
    except OperationalError as e:
        print(f"The error '{e}' occurred")

#### create table

In [16]:
create_users_table = """
CREATE TABLE IF NOT EXISTS applicants (
p_id SERIAL PRIMARY KEY,
program	TEXT,
comments TEXT,
date_added date,
url	TEXT,
status TEXT,
term TEXT,
us_or_international	TEXT,
gpa	FLOAT,
gre	FLOAT,
gre_v FLOAT,
gre_aw FLOAT,
degree TEXT,
llm_generated_program TEXT,
llm_generated_university TEXT
)
"""

execute_query(connection, create_users_table)

#### insert records

In [18]:
# check datatype of the json data file
for key, value in applicant_data[0].items():
    print(key, type(value))

program <class 'str'>
university_raw <class 'str'>
program_raw <class 'str'>
comments <class 'str'>
date_added <class 'str'>
url <class 'str'>
status <class 'str'>
acceptance_date <class 'str'>
rejection_date <class 'NoneType'>
term <class 'str'>
US/International <class 'str'>
Degree <class 'str'>
GPA <class 'NoneType'>
GRE <class 'NoneType'>
GRE V <class 'NoneType'>
GRE AW <class 'NoneType'>
llm-generated-program <class 'str'>
llm-generated-university <class 'str'>


In [19]:
def format_text(value):
    """
    The current value: 
    1.if the value is None,it returns "Null"
    2. replace ' with '' (SQL needs it)
    3. the whole thing quoted with ' ' (SQL needs it)
    """
    if value is None:
        return "NULL"
    return "'" + str(value).replace("'", "''") + "'"


def format_float(value):
    """
    1.if the value is None,it returns "Null"
    2. extract float from json, then convert to string for SQL insertion
    """
    if value is None or value == "":
        return "NULL"
    return str(float(value))


# def create_program(applicant):
#     university = applicant.get("llm-generated-university")
#     program_name = applicant.get("llm-generated-program")

#     if university is None and program_name is None:
#         return "NULL"

#     if university is None:
#         full_program = program_name
#     elif program_name is None:
#         full_program = university
#     else:
#         full_program = program_name + ", " + university

#     return format_text(full_program)


def insert_applicants(connection, applicant_data):
    for applicant in applicant_data:
        insert_query = f"""
        INSERT INTO applicants (
            program,
            comments,
            date_added,
            url,
            status,
            term,
            us_or_international,
            gpa,
            gre,
            gre_v,
            gre_aw,
            degree,
            llm_generated_program,
            llm_generated_university
        )
        VALUES (
            {format_text(applicant.get("program"))},
            {format_text(applicant.get("comments"))},
            {format_text(applicant.get("date_added"))},
            {format_text(applicant.get("url"))},
            {format_text(applicant.get("status"))},
            {format_text(applicant.get("term"))},
            {format_text(applicant.get("US/International"))},
            {format_float(applicant.get("GPA"))},
            {format_float(applicant.get("GRE"))},
            {format_float(applicant.get("GRE V"))},
            {format_float(applicant.get("GRE AW"))},
            {format_text(applicant.get("Degree"))},
            {format_text(applicant.get("llm-generated-program"))},
            {format_text(applicant.get("llm-generated-university"))}
        );
        """

        execute_query(connection, insert_query)


insert_applicants(connection, applicant_data)

#### test connection and Read query 

In [20]:
# Note: the two functions are different: 
# execute_query ---- runs query, does not fetch data
# execute_read_query ---- runs SELECT query, fetches and returns data

def execute_read_query(connection, query):
    cursor = connection.cursor()
    result = None
    try:
        cursor.execute(query)
        result = cursor.fetchall()
        return result
    except OperationalError as e:
        print(f"The error '{e}' occurred")


select_applicants = "SELECT * FROM applicants limit 10"
applicants = execute_read_query(connection, select_applicants)

for applicant in applicants:
    print(applicant)

(1, 'Environmental Economics, Stockholm University', '', datetime.date(2026, 5, 31), 'https://www.thegradcafe.com/result/1020288', 'Accepted on May 31', 'Fall 2026', 'International', None, None, None, None, 'PhD', 'Environmental Economics', 'Stockholm University')
(2, 'Global Health, Meharry Medical College', '', datetime.date(2026, 5, 30), 'https://www.thegradcafe.com/result/1020287', 'Accepted on May 29', 'Fall 2026', 'American', 3.9, None, None, None, 'PhD', 'Global Health', 'Meharry Medical College')
(3, 'Public Policy, University of Massachusetts', '', datetime.date(2026, 5, 29), 'https://www.thegradcafe.com/result/1020286', 'Wait Listed on May 29', 'Fall 2026', 'International', None, None, None, None, 'PhD', 'Public Policy', 'University of Massachusetts')
(4, 'Engineering Science, University of Oxford', "Applying as a (home student) Oxford undergrad. Didn't receive the offer until super late...!", datetime.date(2026, 5, 29), 'https://www.thegradcafe.com/result/1020285', 'Accepted

In [21]:
total_applicants = "SELECT count(*) FROM applicants"
tot = execute_read_query(connection, total_applicants)
print(tot)

[(36000,)]
